In [2]:
from __future__ import annotations
import re
import difflib
from collections import defaultdict
from dataclasses import dataclass
from typing import Iterable

import spacy
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

In [14]:
_nlp = spacy.blank("en")
_nlp.add_pipe('sentencizer')

In [20]:
def load_paragraphs(text) -> list[str]:
    """Split a plain-text file into paragraphs on blank lines."""
    # text = open(path, encoding="utf-8").read()
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    return paragraphs


def paragraph_to_sentences(paragraph: str) -> list[list[str]]:
    """Tokenize a paragraph into a list of sentences, each a list of
    lowercased word tokens (punctuation tokens dropped)."""
    doc = _nlp(paragraph)
    sentences = []
    for sent in doc.sents:
        tokens = [t.text.lower() for t in sent if t.is_alpha]
        if tokens:
            sentences.append(tokens)
    return sentences


def ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    """Slide a window of size n over tokens, one word at a time."""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

In [6]:
@dataclass
class TrigramCorpus:
    paragraphs: list[str]
    # trigram -> set of paragraph ids it appears in
    trigram_index: dict[tuple[str, ...], set[int]]
    # paragraph id -> set of trigrams it contains
    paragraph_trigrams: dict[int, set[tuple[str, ...]]]


def build_trigram_corpus(paragraphs: list[str], n: int = 3) -> TrigramCorpus:
    trigram_index: dict[tuple[str, ...], set[int]] = defaultdict(set)
    paragraph_trigrams: dict[int, set[tuple[str, ...]]] = {}

    for pid, para in enumerate(paragraphs):
        sents = paragraph_to_sentences(para)
        para_grams: set[tuple[str, ...]] = set()
        for sent in sents:
            para_grams.update(ngrams(sent, n))
        paragraph_trigrams[pid] = para_grams
        for g in para_grams:
            trigram_index[g].add(pid)

    return TrigramCorpus(paragraphs, dict(trigram_index), paragraph_trigrams)


def frequent_trigrams(corpus: TrigramCorpus, min_paragraphs: int = 2):
    """Trigrams appearing in at least `min_paragraphs` distinct paragraphs."""
    return {g: pids for g, pids in corpus.trigram_index.items()
            if len(pids) >= min_paragraphs}


def mine_cooccurring_trigram_sets(
    corpus: TrigramCorpus,
    min_paragraphs: int = 2,
    min_support: float | None = None,
):
    """
    Frequent-itemset mining over trigrams, treating each paragraph as a
    transaction. Returns itemsets of trigrams that recur *together*
    across paragraphs -- e.g. the article's example where 'a description
    of', 'now a description', 'this is now', and 'is now a' all co-occur
    in three different paragraphs.
    """
    freq = frequent_trigrams(corpus, min_paragraphs=min_paragraphs)
    if not freq:
        return pd.DataFrame(columns=["support", "itemsets"])

    # Restrict each paragraph's transaction to only the pre-filtered
    # frequent trigrams -- keeps the itemset search space tractable on
    # a novel-length text.
    transactions = []
    for pid in corpus.paragraph_trigrams:
        items = corpus.paragraph_trigrams[pid] & freq.keys()
        transactions.append([" ".join(g) for g in items])

    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df = pd.DataFrame(te_ary, columns=te.columns_)

    n_paras = len(corpus.paragraphs)
    if min_support is None:
        min_support = min_paragraphs / n_paras

    itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)
    itemsets = itemsets[itemsets["itemsets"].apply(len) > 1]  # co-occurring SETS
    return itemsets.sort_values("support", ascending=False).reset_index(drop=True)


In [7]:
def find_long_verbatim_repeats(paragraphs: list[str], n: int = 36):
    """
    Slide an n-word window across each paragraph and find n-grams that
    recur in more than one location (paragraph, position). Long exact
    matches like this are how the original study surfaced the two
    ~500-word verbatim-duplicated sections.
    """
    index: dict[tuple[str, ...], list[tuple[int, int]]] = defaultdict(list)

    for pid, para in enumerate(paragraphs):
        tokens = [t for sent in paragraph_to_sentences(para) for t in sent]
        for i in range(len(tokens) - n + 1):
            g = tuple(tokens[i:i + n])
            index[g].append((pid, i))

    return {g: locs for g, locs in index.items() if len(locs) >= 2}


def find_longest_verbatim_match(paragraphs: list[str], para_a: int, para_b: int):
    """
    Alternative to picking an n up front: use difflib to find the
    longest contiguous matching block of tokens between two paragraphs
    directly. Useful once find_long_verbatim_repeats() has flagged a
    candidate pair.
    """
    tokens_a = [t for sent in paragraph_to_sentences(paragraphs[para_a]) for t in sent]
    tokens_b = [t for sent in paragraph_to_sentences(paragraphs[para_b]) for t in sent]
    matcher = difflib.SequenceMatcher(None, tokens_a, tokens_b, autojunk=False)
    match = matcher.find_longest_match(0, len(tokens_a), 0, len(tokens_b))
    matched_words = tokens_a[match.a: match.a + match.size]
    return matched_words, match

In [8]:
with open('../corpuses/wollstonecraft-corpus/maria.txt') as m:
    maria = m.read()

In [9]:
def clean_gutenberg_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    start_pattern = r"\*\*\*\s*START OF.*?\*\*\*"
    end_pattern = r"\*\*\*\s*END OF.*?\*\*\*"

    start_match = re.search(start_pattern, text, re.DOTALL | re.IGNORECASE)
    end_match = re.search(end_pattern, text, re.DOTALL | re.IGNORECASE)

    if not start_match or not end_match:
        raise ValueError("Could not locate Gutenberg boundaries.")

    cleaned = text[start_match.end():end_match.start()]

    return cleaned.strip()

In [10]:
maria = clean_gutenberg_text('../corpuses/wollstonecraft-corpus/maria.txt')

In [11]:
def unwrap_gutenberg_paragraphs(text):
    """
    Remove Gutenberg line wrapping while preserving paragraphs.
    """

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Split into paragraphs
    paragraphs = re.split(r"\n\s*\n+", text)

    cleaned = []

    for para in paragraphs:
        # Replace internal newlines with spaces
        para = re.sub(r"\s*\n\s*", " ", para)

        # Collapse repeated spaces
        para = re.sub(r"\s+", " ", para)

        cleaned.append(para.strip())

    return "\n\n".join(cleaned)

In [18]:
maria = unwrap_gutenberg_paragraphs(maria)
maria[:5000]

'MARIA\n\nor The Wrongs of Woman\n\nby MARY WOLLSTONECRAFT\n\n(1759-1797)\n\nAfter the edition of 1798\n\nCONTENTS\n\nMARIA PREFACE AUTHOR’S PREFACE CHAPTER 1 CHAPTER 2 CHAPTER 3 CHAPTER 4 CHAPTER 5 CHAPTER 6 CHAPTER 7 CHAPTER 8 CHAPTER 9 CHAPTER 10 CHAPTER 11 CHAPTER 12 CHAPTER 13 CHAPTER 14 APPENDIX CHAPTER 15 CHAPTER 16 CHAPTER 17 CONCLUSION\n\nMARIA\n\nor\n\nThe Wrongs of Woman\n\nPREFACE\n\nThe public are here presented with the last literary attempt of an author, whose fame has been uncommonly extensive, and whose talents have probably been most admired, by the persons by whom talents are estimated with the greatest accuracy and discrimination. There are few, to whom her writings could in any case have given pleasure, that would have wished that this fragment should have been suppressed, because it is a fragment. There is a sentiment, very dear to minds of taste and imagination, that finds a melancholy delight in contemplating these unfinished productions of genius, these sketche

In [21]:
corpus = build_trigram_corpus(load_paragraphs(maria))

In [22]:
freq = frequent_trigrams(corpus, min_paragraphs=2)

In [23]:
freq

{('wrongs', 'of', 'woman'): {1, 9, 19, 27},
 ('the', 'wrongs', 'of'): {1, 9, 19, 27},
 ('filled', 'up', 'in'): {11, 16},
 ('that', 'would', 'have'): {11, 35, 436},
 ('should', 'have', 'been'): {11, 464},
 ('there', 'is', 'a'): {11, 112},
 ('of', 'a', 'world'): {11, 289},
 ('the', 'manners', 'of'): {11, 112},
 ('because', 'it', 'is'): {11, 283},
 ('if', 'they', 'had'): {11, 69, 186, 418},
 ('with', 'the', 'greatest'): {11, 331},
 ('they', 'had', 'been'): {11, 132, 145, 441},
 ('it', 'is', 'a'): {11, 283, 285},
 ('of', 'taste', 'and'): {11, 112},
 ('in', 'my', 'mind'): {12, 169},
 ('some', 'of', 'the'): {12, 60, 74, 120, 157, 171, 342, 406, 441, 447},
 ('to', 'a', 'friend'): {12, 25},
 ('communicated', 'her', 'manuscript'): {12, 25},
 ('ought', 'to', 'be'): {12, 330},
 ('was', 'far', 'from'): {12, 34},
 ('a', 'letter', 'to'): {12, 318, 433},
 ('in', 'some', 'degree'): {12, 415, 461},
 ('avail', 'myself', 'of'): {12, 349},
 ('which', 'i', 'had'): {12,
  113,
  115,
  137,
  140,
  154,
  